# Haja Coração — Pipeline BPM com AQE + otimização manual

Este notebook aplica ao dataset de batimentos a mesma ideia do exemplo de pipeline otimizado: **otimização manual + AQE**.

- manual: selecionar/filtrar cedo e usar `broadcast` para uma dimensão pequena;
- automática: AQE ligado;
- evitar `repartition()` sem necessidade;
- observar `Exchange`, stages e DAG na Spark UI.

A entrada é o XLSX gerado automaticamente pelo `gerador_batimentos.py`. O XLSX é convertido para CSV somente para ingestão; o tratamento é feito pelo Spark.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, avg, stddev, min, max, count, sum as spark_sum, when, round as spark_round
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType
import pandas as pd
import os

spark = (SparkSession.builder
    .appName("HajaCoracao-BPM-AQE-Otimizado")
    .config("spark.ui.port", "4040")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "6")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Spark UI: http://localhost:4040")
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))


## 1. Leitura do XLSX

O Spark SQL não lê XLSX com `spark.read.csv/parquet`. Por isso o notebook usa pandas **somente na entrada**, convertendo a planilha para CSV. Em um cenário maior, a origem ideal seria CSV/JSON/Parquet ou outro formato distribuído.


In [ ]:
XLSX = "/home/ubuntu/dados_batimentos.xlsx"
CSV = "/home/ubuntu/dados_batimentos.csv"

if not os.path.exists(CSV):
    pd.read_excel(XLSX, sheet_name="Dados Batimentos").to_csv(CSV, index=False)

schema = StructType([
    StructField("messageId", IntegerType(), False),
    StructField("deviceId", StringType(), False),
    StructField("heartRate", DoubleType(), False),
    StructField("heartRateTarget", DoubleType(), False),
    StructField("activityState", IntegerType(), False),
    StructField("activityLabel", StringType(), False),
    StructField("bpmAlert", BooleanType(), False),
    StructField("timestamp", StringType(), False),
    StructField("deviceIndex", IntegerType(), False),
    StructField("timeSinceStart", StringType(), False)
])

bpm = spark.read.option("header", True).schema(schema).csv(CSV)
bpm.printSchema()
bpm.show(5, truncate=False)


## 2. Otimização manual: projeção e filtro antes do join

O `select` reduz as colunas que entram nas etapas seguintes. Os filtros eliminam registros inválidos o mais cedo possível.

Isso segue a lógica de **Projection Pruning** e **Predicate Pushdown** apresentada no material: reduzir dados antes de operações caras.


In [ ]:
bpm_filtrado = (bpm
    .select("messageId", "deviceId", "heartRate", "heartRateTarget",
            "activityState", "activityLabel", "bpmAlert", "timestamp")
    .filter(col("heartRate").isNotNull())
    .filter((col("heartRate") >= 30) & (col("heartRate") <= 220))
)

print("Registros após tratamento:", bpm_filtrado.count())
bpm_filtrado.show(5, truncate=False)


## 3. Broadcast: dimensão pequena de dispositivos

Para demonstrar a técnica sem inventar um cenário artificial de tabela grande, modelamos os metadados dos 6 dispositivos como uma pequena dimensão.

`broadcast(dim_devices)` envia essa tabela pequena aos executors e permite que o lado grande faça o join sem o shuffle que seria necessário em um join baseado em redistribuição.

Com apenas 500 registros, o ganho de tempo será pequeno ou até imperceptível; o objetivo é demonstrar a estratégia que escala para uma tabela BPM grande + dimensão pequena.


In [ ]:
dim_devices = spark.createDataFrame([
    ("device-01", 1), ("device-02", 2), ("device-03", 3),
    ("device-04", 4), ("device-05", 5), ("device-06", 6)
], ["deviceId", "deviceIndex"])

bpm_completo = bpm_filtrado.join(
    broadcast(dim_devices),
    on="deviceId",
    how="inner"
)

bpm_completo.explain("formatted")


## 4. AQE + agregação

AQE está ligado. O `groupBy` é uma operação **wide**, pois os registros de uma mesma chave podem estar em partições diferentes. Portanto, é esperado encontrar `Exchange` no plano físico.

Não usamos `repartition()` manualmente: não há motivo para criar um shuffle adicional antes da agregação.


In [ ]:
resultado = (bpm_completo
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
    .groupBy("deviceId", "activityLabel")
    .agg(
        count("*").alias("total_registros"),
        spark_round(avg("heartRate"), 2).alias("media_bpm"),
        spark_round(stddev("heartRate"), 2).alias("desvio_bpm"),
        min("heartRate").alias("min_bpm"),
        max("heartRate").alias("max_bpm"),
        spark_sum(when(col("bpmAlert") == True, 1).otherwise(0)).alias("alertas")
    )
)

resultado.explain("formatted")
resultado.show(20, truncate=False)


## 5. Plano físico e geração de código

Use o `formatted` para registrar `Exchange`, `BroadcastExchange` e a estratégia de join. Use `codegen` para observar a geração de código pelo Spark.

A comparação conceitual é:
**código PySpark → plano lógico → plano otimizado → plano físico → execução/codegen**.


In [ ]:
print("=== PLANO FÍSICO ===")
resultado.explain("formatted")

print("\n=== CODEGEN ===")
resultado.explain("codegen")

print("\n=== CONFIGURAÇÕES ===")
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))


## 6. Resultado tratado

Parquet é usado como saída porque preserva tipos e é adequado para leitura colunar pelo Spark.

A ordenação é feita apenas para apresentação. Ela é uma operação potencialmente wide; portanto, não é necessária para a geração do resultado analítico e não deve ser adicionada ao pipeline apenas para “organizar” os dados.


In [ ]:
OUTPUT = "/home/ubuntu/processed_bpm_parquet"

resultado.write.mode("overwrite").parquet(OUTPUT)
print("Saída:", OUTPUT)


## 7. Evidências para o relatório

Na EC2, acesse a Spark UI por túnel SSH em `http://localhost:4040`.

Na UI:
1. **Jobs** → abra o job gerado pela ação;
2. capture **DAG Visualization**;
3. abra **Stages**;
4. registre tempo, número de tasks, Shuffle Read e Shuffle Write;
5. no `explain("formatted")`, procure `Exchange` e `BroadcastExchange`.

Não invente métricas: os valores do relatório devem ser os observados na execução.


In [ ]:
print("Spark UI continua disponível enquanto esta SparkSession estiver ativa.")
print("Não execute spark.stop() se quiser explorar a UI durante a sessão.")
